# Multi level Perceptron (MLP): Using Numpy and Pandas

* Develop and train a MLP model to accurately classify MNIST data. 
* Use one hidden layer, one input layer and one output layer. 
* Test and choose an appropriate activation function, Optimizer and loss function. 
* Implement forward-propagation and Back-propagation.

In [7]:
import sys
print(sys.executable)

/home/bunny/Desktop/MLP/.venv/bin/python


In [8]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt

## Loading MNIST data
Load data and mention data properties

In [9]:
# Loading test and train data
train_data_input = pd.read_csv('MNIST_CSV/mnist_train.csv', header=None)
test_data_input = pd.read_csv('MNIST_CSV/mnist_test.csv', header=None)

In [33]:
# Creating validation set

val_size = 10000

n = train_data_input.shape[0]

perm = np.random.permutation(n)

train_data_input = train_data_input.iloc[perm]

train_data_final = train_data_input[val_size:]

validation_final = train_data_input[:val_size]

In [11]:
# Data properties

print('Number of training samples : ', n)

print('Data shape : ', train_data_input.shape)

print('Image length : ', train_data_input.iloc[1,1:].shape)

print('Feature Data type : ', type(train_data_input.iloc[1,1]))

print('Class Data type : ', type(train_data_input.iloc[0,0]))


Number of training samples :  60000
Data shape :  (60000, 785)
Image length :  (784,)
Feature Data type :  <class 'numpy.int64'>
Class Data type :  <class 'numpy.int64'>


In [12]:
# Separating Class and Normalizing data from -0.5 to 0.5

x_train = (train_data_input.iloc[:,1:] / 255).to_numpy() - 0.5

y_train = train_data_input.iloc[:,0].to_numpy()

x_test = (test_data_input.iloc[:,1:]  / 255).to_numpy() - 0.5

y_test = test_data_input.iloc[:,0].to_numpy()

x_val = (validation_final.iloc[:,1:]  / 255).to_numpy() - 0.5

y_val = validation_final.iloc[:,0].to_numpy()

print(f'Minimum and Maximum feature values: {x_train.min()}, {x_train.max()}')

Minimum and Maximum feature values: -0.5, 0.5


# Defining parameters

Initializing parameters and storing them in a dictionary

In [13]:
# Defining parameters dictionary
param = {
    'fc1_weights' : None,
    'fc1_bias' : None,
    'fc2_weights' : None,
    'fc2_bias' : None,
    'output_weights' : None,
    'output_bias' : None
}

def initialize_parameters ():
    # Fixing seed
    np.random.seed(42)

    # Initializing weight for Input layer
    param['fc1_weights'] = np.random.randn(784, 256) * np.sqrt(2/784)

    # Initializing bias for Input layer
    param['fc1_bias'] = np.ones((1, 256)) * 0.01

    # Initializing weight for FC1 layer
    param['fc2_weights'] = np.random.randn   (256,128) * np.sqrt(2/256)

    #Initializing bias for FC1 layer
    param['fc2_bias'] = np.ones((1,128)) *.01

    # Initializing weight for FC2 layer
    param['output_weights'] = np.random.randn(128,10) * np.sqrt(2/128)

    #Initializing bias for FC2 layer
    param['output_bias'] = np.ones((1,10)) * 0.01

initialize_parameters()

## Building the FC1 layer

* Building the first fully connected layer of MPL. Using Weight and bias matrices to expand
* Initializing weight with random values to break neuron symmetry



In [14]:


# Input layer function
def forward_fc1(input_array = None, weights = None, bias = None):

    output_matrix = np.matmul(input_array , weights) + bias

    return output_matrix

fc1_output = forward_fc1 (
    x_train, 
    param['fc1_weights'], 
    param['fc1_bias']
    )


# Adding activation layer

Introducing non linearity in neurons through hidden layer to make model functionally strong. Using ReLU, Sigmoid and Tanh as activation functions. Compressing the layer to 512 neurons.

In [15]:
# Creating a function for activation layer

# Activation layer function 
def Layer_activation(input_matrix = None, activation = 'ReLU'):

    if activation == 'ReLU':

        output_matrix = np.maximum(0, input_matrix)
    
        return output_matrix
    
    if activation == 'Leaky ReLU':

        output_matrix = np.maximum(0.01 * input_matrix, input_matrix)
    
        return output_matrix

Activation_output =  Layer_activation (fc1_output)


# Building FC2 layer

* Building a second hidden fully connected layer to ensure gradual compression of neurons.
* Initializing weight with random values to break neuron symmetry


In [16]:
# Layer_2 is a fully connected perceptron layer consisting of 256 neurons

# Fc2 layer function
def forward_fc2 (input_matrix = None, weights = None, bias = None):
    
    output_matrix = np.matmul( input_matrix , weights) + bias

    return output_matrix

fc2_output = forward_fc2 (
    Activation_output, 
    param['fc2_weights'], 
    param['fc2_bias']
    )



# Activation layer

Introducing another non linearity in neurons through hidden layer to make model functionally strong. Using ReLU, Sigmoid and Tanh as activation functions. 

Z3 = ReLU( ReLU( XW1 ) W2 ) W3

In [17]:
# Creating a function for activation layer

Activation_output_2 =  Layer_activation (fc2_output)

# Output Layer

Output layer compress the FC2 output further into 10 classes (digits)

In [18]:

# Output layer function 
def forward_output (input_matrix = None, activation = 'softmax', weights = None, bias = None):
    
    output_matrix = np.matmul( input_matrix , weights ) + bias    

    if activation == 'sigmoid': 

        output_matrix = 1 / ( 1 + np.exp(-output_matrix) )

    if activation == 'softmax':

        z = output_matrix - output_matrix.max( axis=1, keepdims = True)

        exp_z = np.exp(z)

        output_matrix = exp_z / exp_z.sum( axis=1, keepdims = True)

    return output_matrix

predictions = forward_output(
    Activation_output_2,
    'softmax',
    param['output_weights'],
    param['output_bias']
)


# One hot encoding for target variables for all training data
Converting the target array into 2d one hot encoded array for easier loss calculation 

In [19]:
y_train_ohe = np.zeros((y_train.shape[0],10))

y_train_ohe[np.arange(y_train.shape[0]),y_train] =1

print(y_train_ohe.shape)

(60000, 10)


# Variance analysis

Variance should neither shrink or increase by order of a magnitude through each layer.

In [20]:
# Comparing variance

print('Input variance: ', np.mean(np.var(x_train, axis=0)))
print('FC1 Layer variance: ', np.mean(np.var(fc1_output, axis=0)))
print('Activation Layer 1 variance: ', np.mean(np.var(Activation_output, axis=0)))
print('FC2 Layer variance: ', np.mean(np.var(fc2_output, axis=0)))
print('Activation Layer variance: ', np.mean(np.var(Activation_output_2, axis=0)))      



Input variance:  0.06725132078460055
FC1 Layer variance:  0.13573078085209025
Activation Layer 1 variance:  0.056188870835491074
FC2 Layer variance:  0.10894945535543159
Activation Layer variance:  0.042053235893994914


## Conclusions of variance analysis

* ReLU activation layer only activated when input range was changed from (0,1) to (0.5,0.5)
* ReLU Reduces variance to 0 when bias is zero. Bias needs to be small positive value
* Leaky ReLU performs better is preventing the variance from turning to 0
* Zero centered input and initial weights are essential for variance to not collapse in the activation (ReLU) layer

# Loss function
Using cross entropy as loss function for simple loss gradient function.
* Loss in cross entropy is always positive number as predictions range from (0, 1)

In [21]:
# Implementing cross entropy loss function

def cross_entropy (input_array = None , target = None):

    epsilon = 1e-8

    loss = -np.sum(target * np.log(input_array + epsilon), axis=1) 
    
    return loss

loss_initial = cross_entropy(predictions, y_train_ohe)

# Mean loss and accuracy
print( np.mean(loss_initial))


2.4619276262896173


# Gradient calculations




## Backpropagation — Gradients of Weights and Biases w.r.t Loss

Gradient is defined as the **rate of change of loss with respect to a variable**. Using the chain rule, gradients are propagated in reverse through the network: **Output Layer → FC2 → FC1**.

Normalizing gradients of weights and biases to ensure gradient doesn't get impacted by large sample size. 
- `dz` = dz / sample size

**Notation:**
- `z` = pre-activation output
- `a` = post-activation output (after ReLU)
- `dz` = dL/dz &nbsp;|&nbsp; `dw` = dL/dW &nbsp;|&nbsp; `db` = dL/db &nbsp;|&nbsp; `da` = dL/da

In [22]:
# Initializing gradient parameters

gradient_params = {
    'dw3' : None,
    'dw2' : None,
    'dw1' : None,
    'db3' : None,
    'db2' : None,
    'db1' : None
}

In [23]:

def gradient (batch_data = None, fc1_output = None, Activation_output = None, fc2_output = None, Activation_output_2 = None, predictions = None, One_hot_encoded_train = None):

    w = predictions.shape[0]
    
    # --------------------------------------------------------------
    # OUTPUT LAYER (FC3)
    # Loss: cross-entropy | Activation: softmax
    # dL/dz3 = predictions - y  (softmax + cross-entropy combined)
    # --------------------------------------------------------------

    # Gradient of loss w.r.t. output layer pre-activation
    dz3 = predictions - One_hot_encoded_train


    # Gradient of loss w.r.t. output layer weights
    # dL/dW3 = A2^T @ dz3
    dw3 = np.transpose(Activation_output_2) @ dz3

    # Normalizing gradients
    dw3 /= w

    # Storing gradient in dictionary
    gradient_params['dw3'] = dw3

    # Gradient of loss w.r.t. output layer bias
    db3 = np.sum(dz3, axis=0, keepdims=True)

    # Normalizing gradients
    db3 /= w

    # Storing gradient in dictionary
    gradient_params['db3'] = db3


    # --------------------------------------------------------------
    # SECOND HIDDEN LAYER (FC2)
    # Activation: ReLU  →  ReLU gradient = 1 if z > 0, else 0
    # --------------------------------------------------------------

    # Gradient of loss w.r.t. FC2 post-activation (backdrop through output weights)
    # dL/dA2 = dz3 @ W3^T
    da2 = dz3 @ np.transpose(param['output_weights'])

    # Gradient of loss w.r.t. FC2 pre-activation (backprop through ReLU)
    # ReLU derivative: pass gradient only where FC2 output was positive
    dz2 = da2 * (fc2_output > 0)

    # Gradient of loss w.r.t. FC2 weights
    # dL/dW2 = A1^T @ dz2
    dw2 = np.transpose(Activation_output) @ dz2

    # Normalizing gradients
    dw2 /= w

    # Storing gradient in dictionary
    gradient_params['dw2'] = dw2

    # Gradient of loss w.r.t. FC2 bias
    db2 = np.sum(dz2, axis=0, keepdims=True)

    # Normalizing gradients
    db2 /= w

    # Storing gradient in dictionary
    gradient_params['db2'] = db2

    # --------------------------------------------------------------
    # FIRST HIDDEN LAYER (FC1)
    # Activation: ReLU  →  ReLU gradient = 1 if z > 0, else 0
    # --------------------------------------------------------------

    # Gradient of loss w.r.t. FC1 post-activation (backprop through FC2 weights)
    # dL/dA1 = dz2 @ W2^T
    da1 = dz2 @ np.transpose(param['fc2_weights'])

    # Gradient of loss w.r.t. FC1 pre-activation (backprop through ReLU)
    # ReLU derivative: pass gradient only where FC1 output was positive
    dz1 = da1 * (fc1_output > 0)

    # Gradient of loss w.r.t. FC1 weights
    # dL/dW1 = X^T @ dz1
    dw1 = np.transpose(batch_data) @ dz1

    # Normalizing gradients
    dw1 /= w

    # Storing gradient in dictionary
    gradient_params['dw1'] = dw1

    # Gradient of loss w.r.t. FC1 bias
    db1 = np.sum(dz1, axis=0, keepdims=True)

    # Normalizing gradients
    db1 /= w

    # Storing gradient in dictionary
    gradient_params['db1'] = db1



# Gradient decent

Changing the weights according to the gradients and comparing loss function.

In [24]:
    # learning rate is a hyperparameter to control rate of change of step size
    
def gradient_decent (learning_rate = 0.01):

    # --------------------------------------------------------------
    # WEIGHTS GRADIENT DECENT
    # STEP SIZE = learning_rate * Gradient
    # --------------------------------------------------------------

    param['output_weights'] -= gradient_params['dw3'] * learning_rate

    param['fc2_weights'] -= gradient_params['dw2'] * learning_rate

    param['fc1_weights'] -= gradient_params['dw1'] * learning_rate

    # --------------------------------------------------------------
    # BIASES GRADIENT DECENT
    # STEP SIZE = learning_rate * Gradient
    # --------------------------------------------------------------

    param['output_bias'] -= gradient_params['db3'] * learning_rate

    param['fc2_bias'] -= gradient_params['db2'] * learning_rate

    param['fc1_bias'] -= gradient_params['db1'] * learning_rate


In [25]:
# Calculating and comparing Loss

def loss_calculation ():

    fc1_output = forward_fc1 (x_train, param['fc1_weights'], param['fc1_bias'] )

    Activation_output_1 =  Layer_activation (fc1_output)

    fc2_output = forward_fc2 (Activation_output_1, param['fc2_weights'], param['fc2_bias'])

    Activation_output_2 =  Layer_activation (fc2_output)

    predictions = forward_output (Activation_output_2, 'softmax', param['output_weights'], param['output_bias'])

    loss = cross_entropy( predictions, y_train_ohe)

    return loss

# Mean Loss and Accuracy comparison

Loss = np.mean(loss_calculation())

print(Loss)


2.4619276262896173


# Implementing multiple passes

Performing multiple epochs to reduce loss

In [26]:
def train_step (batch_data = None, label_data = None, One_hot_encoded_train = None, learning_rate = 0.01):

    fc1 = forward_fc1 (batch_data, param['fc1_weights'], param['fc1_bias'])

    Activation =  Layer_activation (fc1)

    fc2 = forward_fc2 (Activation, param['fc2_weights'], param['fc2_bias'])

    Activation_2 =  Layer_activation (fc2)

    final_output = forward_output (Activation_2, 'softmax', param['output_weights'], param['output_bias'])
    
    gradient( batch_data, fc1, Activation, fc2, Activation_2, final_output, One_hot_encoded_train)

    gradient_decent (learning_rate)

    Loss = np.mean(cross_entropy( final_output, One_hot_encoded_train))

    predicted_labels = np.argmax(final_output, axis = 1 )

    Acc = np.mean( predicted_labels == label_data)

    return Loss, Acc

initialize_parameters()

for epoch in range(200):

    if epoch < 7:
        learning_rate = 0.09

    elif epoch < 12:
        learning_rate = 0.08
        
    elif epoch < 20:
        learning_rate = 0.075

    elif epoch < 30:
        learning_rate = 0.065

    elif epoch < 40:
        learning_rate = 0.06

    elif epoch < 150:
        learning_rate = 0.053

    else:
        learning_rate = 0.03

    Loss, Acc = train_step (x_train, y_train, y_train_ohe, learning_rate)

    print(f"Epoch {epoch}: Loss={Loss}, Accuracy={Acc}")
    

Epoch 0: Loss=2.4619276262896173, Accuracy=0.0798
Epoch 1: Loss=2.2157088966936405, Accuracy=0.23521666666666666
Epoch 2: Loss=2.096096325384845, Accuracy=0.3236
Epoch 3: Loss=1.9934563081549592, Accuracy=0.43033333333333335
Epoch 4: Loss=1.89912522733087, Accuracy=0.46958333333333335
Epoch 5: Loss=1.8118083840235097, Accuracy=0.5382666666666667
Epoch 6: Loss=1.7327941278265944, Accuracy=0.5439
Epoch 7: Loss=1.668876862138463, Accuracy=0.58835
Epoch 8: Loss=1.6314547505715609, Accuracy=0.55215
Epoch 9: Loss=1.602037482369404, Accuracy=0.5716
Epoch 10: Loss=1.6276069861297546, Accuracy=0.5111833333333333
Epoch 11: Loss=1.5721663938842978, Accuracy=0.5449333333333334
Epoch 12: Loss=1.5575556393614287, Accuracy=0.5391166666666667
Epoch 13: Loss=1.3947544889965027, Accuracy=0.6465166666666666
Epoch 14: Loss=1.3187800122928224, Accuracy=0.6676833333333333
Epoch 15: Loss=1.2542482960453907, Accuracy=0.7067333333333333
Epoch 16: Loss=1.217860559466268, Accuracy=0.7008166666666666


KeyboardInterrupt: 

In [49]:
# Loss on validation data

def Accuracy (dataset = "val"):

    if dataset == "train": 
        input_data = x_train
        label_data = y_train
    
    if dataset == "val":
        input_data = x_val
        label_data = y_val

    if dataset == "test":
        input_data = x_test
        label_data = y_test

    fc1_output = forward_fc1 (input_data, param['fc1_weights'], param['fc1_bias'])

    Activation_output =  Layer_activation (fc1_output)

    fc2_output = forward_fc2 (Activation_output, param['fc2_weights'], param['fc2_bias'])

    Activation_output_2 =  Layer_activation (fc2_output)

    predictions = forward_output (Activation_output_2, 'softmax', param['output_weights'], param['output_bias'])

    predicted_labels = np.argmax(predictions, axis=1)

    Acc = np.mean( predicted_labels == label_data)

    return Acc

print ('Accuracy on test data : ', Accuracy("test"))

Accuracy on test data :  0.9792


# Results

- Reached 89.33 % accuracy on training data and 89.91 % accuracy on test data.
- There is no over-fitting

# MINI-BATCH TRAINING

Implementing batch wise training to reduce-

- gradients too smooth
- poor generalization
- slow learning

In [48]:
batch_size = 60

initialize_parameters()

for epoch in range(15):

    perm = np.random.permutation(n)
    
    # perm = np.arange(0, n)

    x_perm = x_train[perm]
    y_perm = y_train[perm]
    y_perm_ohe = y_train_ohe[perm]
    
    for i in range(0, x_train.shape[0], batch_size):

        x_batch = x_perm[i:i+batch_size,:]
        y_batch = y_perm[i:i+batch_size]
        y_train_ohe_batch = y_perm_ohe[i:i+batch_size,:]
        

        learning_rate = 0.09

        loss, acc = train_step (x_batch, y_batch, y_train_ohe_batch, learning_rate)

    Loss = np.mean( loss_calculation())

    Train_Acc = Accuracy("train")

    Val_Acc = Accuracy("val")
    
    print(f"Epoch = {epoch+1}: Loss = {Loss}, Accuracy on training data = {Train_Acc}, Accuracy on Validation data = {Val_Acc}")


Epoch = 1: Loss = 0.1778241260525622, Accuracy on training data = 0.9460333333333333, Accuracy on Validation data = 0.9457
Epoch = 2: Loss = 0.1549263392393333, Accuracy on training data = 0.9539666666666666, Accuracy on Validation data = 0.9525
Epoch = 3: Loss = 0.104518080936741, Accuracy on training data = 0.9680833333333333, Accuracy on Validation data = 0.9686
Epoch = 4: Loss = 0.1147214993113041, Accuracy on training data = 0.9633666666666667, Accuracy on Validation data = 0.9634
Epoch = 5: Loss = 0.06237722730931699, Accuracy on training data = 0.9813, Accuracy on Validation data = 0.9794
Epoch = 6: Loss = 0.05824179862701185, Accuracy on training data = 0.9821833333333333, Accuracy on Validation data = 0.9799
Epoch = 7: Loss = 0.057002450522925185, Accuracy on training data = 0.98155, Accuracy on Validation data = 0.9821
Epoch = 8: Loss = 0.04665730940065384, Accuracy on training data = 0.9853333333333333, Accuracy on Validation data = 0.9842
Epoch = 9: Loss = 0.034602222047166

# Conclusions

- **Batch-wise training significantly improves accuracy** — from 89% up to 97.5%
- **Fewer epochs required**: batch-wise training reaches 97% accuracy in as few as 10 epochs
- **Batch randomization prevents overfitting** *(add detail here if applicable)*
- **Higher learning rates converge faster but risk instability** — in batch-wise training, a learning rate of 0.09 achieves high accuracy quickly but is susceptible to gradient explosion; results are seed-sensitive (works with seed 42, fails on others)